# Notebook 04 — Dataset Final Tratado e Catálogo de Dados

**Projeto Mensal 3 — Tratamento de Dados**  
**Base:** Cargos Vagos e Vacâncias do Poder Executivo Federal Civil — competência 03/2026

---

## Objetivo deste notebook

Este é o **notebook de fechamento** do projeto. Ele consome o `dataset_com_features.csv` gerado pelo Notebook 03 e produz os dois entregáveis finais exigidos pelo enunciado:

1. **`dataset_final_tratado.csv`** (e versão `.xlsx`) — base limpa, transformada, enriquecida e pronta para BI/Data Mining (etapa 6.15).
2. **`catalogo_dados.xlsx`** — documentação técnica de cada coluna do dataset final (etapa 6.16).

## Etapas cobertas

| Etapa | Conteúdo |
|---|---|
| **6.15** | Validações finais, reorganização lógica das colunas, exportação do dataset final |
| **6.16** | Construção do catálogo de dados com 8 atributos por coluna |


## Setup — bibliotecas e configurações

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

# Caminhos do projeto
BASE_DIR       = Path('..').resolve()
DADOS_TRATADOS = BASE_DIR / 'dados_tratados'
DOCUMENTACAO   = BASE_DIR / 'documentacao'

print(f"Base do projeto: {BASE_DIR}")
print(f"Versão pandas:  {pd.__version__}")

Base do projeto: /home/claude/PM3_CargosVagosVacancias_v2
Versão pandas:  3.0.2


## Carga do dataset com features

Importamos o output do Notebook 03 (`dataset_com_features.csv`, 37 colunas).

In [2]:
# Especificação dos dtypes (mesma usada nos notebooks anteriores)
DTYPES_LEITURA = {
    'cod_orgao': str,
    'cod_cargo': str,
    'em_extincao': 'category',
    'nivel': 'category',
    'plano_carreira': 'category',
    'porte_cargo': 'category',
    'faixa_taxa_ocupacao': 'category',
}

df = pd.read_csv(
    DADOS_TRATADOS / 'dataset_com_features.csv',
    dtype=DTYPES_LEITURA,
    parse_dates=['ano_mes']
)

print(f"Dataset carregado:")
print(f"  Linhas:  {df.shape[0]:,}".replace(',', '.'))
print(f"  Colunas: {df.shape[1]}")
print(f"  Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Dataset carregado:
  Linhas:  12.769
  Colunas: 37
  Memória: 6.54 MB


## 1. Validações finais de qualidade (etapa 6.15)

Antes de declarar o dataset como "final", precisamos confirmar que ele atende a **todos os requisitos do enunciado** para a etapa 6.15:

> *"Essa base final deverá estar: limpa; padronizada; sem duplicidades indevidas; com valores ausentes tratados ou justificados; com outliers analisados; com colunas relevantes; com novas variáveis criadas; com nomes de colunas claros; com tipos de dados corretos; pronta para uso em BI, Data Mining ou análise exploratória."*

### 1.1 Checklist de qualidade


In [3]:
# 1.1 Verificações automatizadas
checks = []

# Limpa — sem espaços extras nos campos texto
texto = ['nome_orgao', 'sigla_orgao', 'nome_cargo']
problemas_texto = sum(df[c].str.strip().ne(df[c]).sum() for c in texto)
checks.append(('Sem espaços extras nos campos texto', problemas_texto == 0,
              f'{problemas_texto} ocorrências'))

# Padronizada — todas as colunas em snake_case minúsculo
nao_snake = [c for c in df.columns if c != c.lower() or ' ' in c]
checks.append(('Todas as colunas em snake_case', len(nao_snake) == 0,
              f'{len(nao_snake)} fora do padrão'))

# Sem duplicatas pela chave funcional
dup_chave = df.duplicated(subset=['cod_orgao', 'cod_cargo']).sum()
checks.append(('Sem duplicatas (cod_orgao + cod_cargo)', dup_chave == 0,
              f'{dup_chave} duplicatas'))

# Tipos corretos — tolerante a versões do pandas
# (pandas 2.x: 'object'/'datetime64[ns]'; pandas 3.x: 'str'/'datetime64[us]')
tipos_aceitos = {
    'ano_mes':      ('datetime64',),
    'cod_orgao':    ('object', 'str', 'string'),
    'cod_cargo':    ('object', 'str', 'string'),
    'nivel':        ('category',),
    'em_extincao':  ('category',),
    'qtd_aprovada': ('int64',),
}
tipos_ok = all(
    any(str(df[c].dtype).startswith(t) for t in aceitos)
    for c, aceitos in tipos_aceitos.items()
)
detalhe_tipos = ', '.join(f'{c}={df[c].dtype}' for c in tipos_aceitos)
checks.append(('Tipos de dados corretos', tipos_ok,
              'tolerante a pandas 2.x e 3.x'))

# Nulos esperados/justificados
nulos = df.isnull().sum()
nulos_em_taxas = nulos[['taxa_ocupacao', 'taxa_vacancia_mensal', 'pct_aposentadoria']].sum()
nulos_em_outras = nulos.drop(['taxa_ocupacao', 'taxa_vacancia_mensal',
                              'pct_aposentadoria', 'faixa_taxa_ocupacao']).sum()
checks.append(('Nulos só em colunas onde foi justificado',
              nulos_em_outras == 0,
              f'em taxas: {nulos_em_taxas} (esperado); em outras: {nulos_em_outras}'))

# Shape esperado
checks.append(('Shape do dataset = (12.769, 37)',
              df.shape == (12769, 37),
              f'real: {df.shape}'))

# Imprime o relatório
print(f"{'Critério':<48}{'Status':<10}{'Detalhe':<25}")
print('-' * 83)
for c, ok, detalhe in checks:
    status = '✓ OK' if ok else '✗ FALHA'
    print(f"{c:<48}{status:<10}{detalhe:<25}")

# Asserções estritas — se algo falhar, o notebook para
for c, ok, detalhe in checks:
    assert ok, f"Falha no critério: {c} ({detalhe})"

print('\n✓ Todos os critérios de qualidade foram atendidos.')

Critério                                        Status    Detalhe                  
-----------------------------------------------------------------------------------
Sem espaços extras nos campos texto             ✓ OK      0 ocorrências            
Todas as colunas em snake_case                  ✓ OK      0 fora do padrão         
Sem duplicatas (cod_orgao + cod_cargo)          ✓ OK      0 duplicatas             
Tipos de dados corretos                         ✓ OK      tolerante a pandas 2.x e 3.x
Nulos só em colunas onde foi justificado        ✓ OK      em taxas: 8545 (esperado); em outras: 0
Shape do dataset = (12.769, 37)                 ✓ OK      real: (12769, 37)        

✓ Todos os critérios de qualidade foram atendidos.


### 1.2 Inventário de nulos por coluna (justificativa)

Os nulos remanescentes estão **apenas em duas features derivadas**, ambos justificados pela impossibilidade matemática de calcular a divisão. Uma terceira feature (`taxa_ocupacao`) prevê tratamento de NaN no código, mas neste snapshot específico (03/2026) **nenhuma linha** dispara essa condição — porque a base não inclui cargos com `qtd_aprovada = 0`. O tratamento permanece no código por robustez (para snapshots futuros que possam conter tais casos).


In [4]:
nulos = df.isnull().sum()
print('Colunas com nulos:')
for c, n in nulos.items():
    if n > 0:
        pct = n / len(df) * 100
        print(f"  {c:<30} {n:>5,} ({pct:.1f}%)".replace(',', '.'))

print('\nJustificativa (documentada no Notebook 03, seção 1.2):')
print('  - taxa_ocupacao:        NaN quando qtd_aprovada = 0 (cargo sem aprovação legal)')
print('  - taxa_vacancia_mensal: NaN quando qtd_ocupada = 0 (cargo sem ocupantes)')
print('  - pct_aposentadoria:    NaN quando total_vacancias = 0 (sem vacância no mês)')

Colunas com nulos:
  pct_aposentadoria              7.403 (58.0%)
  taxa_vacancia_mensal           1.142 (8.9%)

Justificativa (documentada no Notebook 03, seção 1.2):
  - taxa_ocupacao:        NaN quando qtd_aprovada = 0 (cargo sem aprovação legal)
  - taxa_vacancia_mensal: NaN quando qtd_ocupada = 0 (cargo sem ocupantes)
  - pct_aposentadoria:    NaN quando total_vacancias = 0 (sem vacância no mês)


**Decisão:** os nulos serão **preservados** no dataset final. Substituí-los por 0 seria semanticamente errado (não é o mesmo que "0% de aposentadoria" — é "indefinido"). Em ferramentas de BI, basta filtrar essas linhas no widget específico que usa a métrica.

## 2. Reordenação lógica das colunas (etapa 6.15)

A ordem atual das colunas reflete a sequência de criação (notebooks 02 e 03). Para o dataset final, reorganizamos em **blocos semânticos** que facilitam a leitura por humanos e o consumo por ferramentas de BI:

| Bloco | Função | Colunas |
|---|---|---|
| **A. Temporal** | Quando | `ano_mes`, `ano`, `mes`, `trimestre` |
| **B. Identificação do órgão** | Onde | `cod_orgao`, `sigla_orgao`, `nome_orgao` |
| **C. Identificação do cargo** | O quê | `cod_cargo`, `nome_cargo`, `plano_carreira`, `nivel`, `em_extincao` |
| **D. Quantitativos brutos** | Stock | `qtd_aprovada`, `qtd_distribuida`, `qtd_ocupada`, `qtd_vaga` |
| **E. Vacâncias por tipo** | Fluxo | as 7 `vac_*` |
| **F. Features derivadas (quant.)** | KPIs | `total_vacancias`, `deficit_nominal`, `taxa_ocupacao`, `taxa_vacancia_mensal`, `pct_aposentadoria`, `diferenca_distribuicao` |
| **G. Features derivadas (cat.)** | Faixas | `porte_cargo`, `faixa_taxa_ocupacao` |
| **H. Versões normalizadas** | Para ML | as 5 `*_robust` |
| **I. Flags** | Metadados | `is_outlier` |


In [5]:
# Definir a ordem final em blocos semânticos
ORDEM_FINAL = [
    # A. Temporal
    'ano_mes', 'ano', 'mes', 'trimestre',
    # B. Identificação do órgão
    'cod_orgao', 'sigla_orgao', 'nome_orgao',
    # C. Identificação do cargo
    'cod_cargo', 'nome_cargo', 'plano_carreira', 'nivel', 'em_extincao',
    # D. Quantitativos brutos
    'qtd_aprovada', 'qtd_distribuida', 'qtd_ocupada', 'qtd_vaga',
    # E. Vacâncias por tipo (em ordem de relevância: aposentadoria primeiro)
    'vac_aposentadoria', 'vac_posse_cargo_inac', 'vac_exoneracao',
    'vac_falecimento', 'vac_demissao', 'vac_promocao', 'vac_readaptacao',
    # F. Features derivadas (quantitativas)
    'total_vacancias', 'deficit_nominal', 'taxa_ocupacao',
    'taxa_vacancia_mensal', 'pct_aposentadoria', 'diferenca_distribuicao',
    # G. Features derivadas (categóricas)
    'porte_cargo', 'faixa_taxa_ocupacao',
    # H. Versões normalizadas
    'qtd_aprovada_robust', 'qtd_ocupada_robust', 'qtd_vaga_robust',
    'total_vacancias_robust', 'deficit_nominal_robust',
    # I. Flags
    'is_outlier',
]

# Sanity check: ordem cobre exatamente todas as colunas (sem perder nem duplicar)
assert set(ORDEM_FINAL) == set(df.columns), \
    f"Inconsistência. Faltam: {set(df.columns) - set(ORDEM_FINAL)}. Sobram: {set(ORDEM_FINAL) - set(df.columns)}"
assert len(ORDEM_FINAL) == len(df.columns), 'Ordem tem duplicatas ou faltantes'

# Aplicar a ordem
df_final = df[ORDEM_FINAL].copy()
print(f"Dataset reordenado: {df_final.shape[1]} colunas, na seguinte sequência:\n")

# Mostrar em blocos
blocos = [
    ('A. Temporal',                     ORDEM_FINAL[0:4]),
    ('B. Identificação do órgão',       ORDEM_FINAL[4:7]),
    ('C. Identificação do cargo',       ORDEM_FINAL[7:12]),
    ('D. Quantitativos brutos',         ORDEM_FINAL[12:16]),
    ('E. Vacâncias por tipo',           ORDEM_FINAL[16:23]),
    ('F. Features derivadas (quant.)',  ORDEM_FINAL[23:29]),
    ('G. Features derivadas (cat.)',    ORDEM_FINAL[29:31]),
    ('H. Versões normalizadas',         ORDEM_FINAL[31:36]),
    ('I. Flags',                        ORDEM_FINAL[36:37]),
]
for nome, cols in blocos:
    print(f"  {nome} ({len(cols)}):")
    for c in cols:
        print(f"    • {c}")

Dataset reordenado: 37 colunas, na seguinte sequência:

  A. Temporal (4):
    • ano_mes
    • ano
    • mes
    • trimestre
  B. Identificação do órgão (3):
    • cod_orgao
    • sigla_orgao
    • nome_orgao
  C. Identificação do cargo (5):
    • cod_cargo
    • nome_cargo
    • plano_carreira
    • nivel
    • em_extincao
  D. Quantitativos brutos (4):
    • qtd_aprovada
    • qtd_distribuida
    • qtd_ocupada
    • qtd_vaga
  E. Vacâncias por tipo (7):
    • vac_aposentadoria
    • vac_posse_cargo_inac
    • vac_exoneracao
    • vac_falecimento
    • vac_demissao
    • vac_promocao
    • vac_readaptacao
  F. Features derivadas (quant.) (6):
    • total_vacancias
    • deficit_nominal
    • taxa_ocupacao
    • taxa_vacancia_mensal
    • pct_aposentadoria
    • diferenca_distribuicao
  G. Features derivadas (cat.) (2):
    • porte_cargo
    • faixa_taxa_ocupacao
  H. Versões normalizadas (5):
    • qtd_aprovada_robust
    • qtd_ocupada_robust
    • qtd_vaga_robust
    • total_vacancia

### 2.1 Amostra do dataset final

In [6]:
print("Primeiras 3 linhas (amostra das colunas mais importantes):\n")
df_final[['ano_mes', 'sigla_orgao', 'nome_cargo', 'nivel',
          'qtd_aprovada', 'qtd_ocupada', 'qtd_vaga',
          'taxa_ocupacao', 'porte_cargo', 'faixa_taxa_ocupacao']].head(3)

Primeiras 3 linhas (amostra das colunas mais importantes):



,ano_mes,sigla_orgao,nome_cargo,nivel,qtd_aprovada,qtd_ocupada,qtd_vaga,taxa_ocupacao,porte_cargo,faixa_taxa_ocupacao
0,2026-03-01,MAPA,AGENTE ADMINISTRATIVO,NI,2,2,0,1.0,Micro (1-2),Completa (90-100%)
1,2026-03-01,MAPA,MEDICO,NS,1,1,0,1.0,Micro (1-2),Completa (90-100%)
2,2026-03-01,MAPA,AUXILIAR OPERACIONAL DE AGROPECUARIA,NAO_INFORMADO,1,1,0,1.0,Micro (1-2),Completa (90-100%)


## 3. Exportação do dataset final (etapa 6.15)

Salvamos em **dois formatos**:

- **CSV** (`dataset_final_tratado.csv`): universal, recomendado para reprodutibilidade e versionamento.
- **XLSX** (`dataset_final_tratado.xlsx`): conveniência para usuários do Excel; documentado no catálogo qual é o canônico.


In [7]:
# 3.1 Exportar CSV
caminho_csv = DADOS_TRATADOS / 'dataset_final_tratado.csv'
df_final.to_csv(caminho_csv, index=False, encoding='utf-8')

print(f"CSV salvo: {caminho_csv}")
print(f"  Tamanho: {caminho_csv.stat().st_size / 1024**2:.2f} MB")

# Releitura de validação
df_check = pd.read_csv(caminho_csv, dtype=DTYPES_LEITURA, parse_dates=['ano_mes'])
assert df_check.shape == df_final.shape, 'Shape mudou após salvar/reler'
print(f"  ✓ Releitura confirma shape ({df_check.shape[0]:,} × {df_check.shape[1]}).".replace(',', '.'))

CSV salvo: /home/claude/PM3_CargosVagosVacancias_v2/dados_tratados/dataset_final_tratado.csv
  Tamanho: 3.48 MB


  ✓ Releitura confirma shape (12.769 × 37).


In [8]:
# 3.2 Exportar XLSX
caminho_xlsx = DADOS_TRATADOS / 'dataset_final_tratado.xlsx'
df_final.to_excel(caminho_xlsx, index=False, sheet_name='Dataset_Final', engine='openpyxl')

print(f"XLSX salvo: {caminho_xlsx}")
print(f"  Tamanho: {caminho_xlsx.stat().st_size / 1024**2:.2f} MB")

XLSX salvo: /home/claude/PM3_CargosVagosVacancias_v2/dados_tratados/dataset_final_tratado.xlsx
  Tamanho: 1.82 MB


## 4. Catálogo de dados (etapa 6.16)

> *"O catálogo serve para documentar o significado de cada coluna. Deverá conter: nome, descrição, tipo, exemplo, origem, tratamento aplicado, indicação se a coluna é original ou criada, uso esperado na análise."*

Construímos o catálogo em duas etapas:
1. **Definir** os 8 atributos para cada uma das 37 colunas (em código, como dicionário).
2. **Validar** que toda coluna do dataset final está no catálogo (e vice-versa).
3. **Exportar** como `catalogo_dados.xlsx` com formatação profissional (cabeçalho destacado, larguras ajustadas).

### 4.1 Definição do catálogo


In [9]:
# Definição do catálogo — uma entrada por coluna
# Estrutura: (nome, descricao, tipo, origem, tratamento, origem_tipo, uso_esperado)
CATALOGO = [
    # ───────── Bloco A. Temporal ─────────
    ('ano_mes',
     'Competência mensal do snapshot (primeiro dia do mês)',
     'datetime64[ns]',
     'Base SEGES — campo ANO_MES (int 202603)',
     'Conversão de int (AAAAMM) para datetime',
     'Original (transformada)',
     'Chave de partição; análise temporal quando concatenada com outras competências'),

    ('ano',
     'Ano da competência, extraído de ano_mes',
     'int64',
     'Derivado de ano_mes',
     'pd.to_datetime().dt.year',
     'Criada',
     'Filtragem em séries históricas; agregações anuais em BI'),

    ('mes',
     'Mês da competência (1-12), extraído de ano_mes',
     'int64',
     'Derivado de ano_mes',
     'pd.to_datetime().dt.month',
     'Criada',
     'Análise de sazonalidade; agrupamento mensal em BI'),

    ('trimestre',
     'Trimestre da competência (1-4), extraído de ano_mes',
     'int64',
     'Derivado de ano_mes',
     'pd.to_datetime().dt.quarter',
     'Criada',
     'Agregação trimestral em dashboards executivos'),

    # ───────── Bloco B. Identificação do órgão ─────────
    ('cod_orgao',
     'Código identificador do órgão no SIORG (5 dígitos com zero à esquerda)',
     'string',
     'Base SEGES — campo ORGAO (int)',
     'Conversão de int para str com zfill(5); renomeado de ORGAO',
     'Original (transformada)',
     'Chave para joins com outras bases federais; dimensão de drill-down'),

    ('sigla_orgao',
     'Sigla oficial do órgão (forma curta)',
     'string',
     'Base SEGES — campo SIGLA_ORGAO',
     'Renomeado para snake_case',
     'Original',
     'Dimensão para rótulos em gráficos e tabelas'),

    ('nome_orgao',
     'Nome oficial completo do órgão',
     'string',
     'Base SEGES — campo NOME_ORGAO',
     'Renomeado para snake_case',
     'Original',
     'Dimensão analítica; rótulo em relatórios formais'),

    # ───────── Bloco C. Identificação do cargo ─────────
    ('cod_cargo',
     'Código identificador do cargo (6 dígitos: 3 plano + 3 cargo)',
     'string',
     'Base SEGES — campo CARGO (int)',
     'Conversão de int para str com zfill(6); renomeado de CARGO',
     'Original (transformada)',
     'Chave; potencial decomposição em cod_plano + cod_cargo_interno'),

    ('nome_cargo',
     'Nome oficial do cargo',
     'string',
     'Base SEGES — campo NOME_CARGO',
     'Renomeado para snake_case',
     'Original',
     'Dimensão; agrupamento por descrição'),

    ('plano_carreira',
     'Plano de carreira ao qual o cargo está vinculado',
     'category',
     'Base SEGES — campo PLANO_CARREIRA',
     '363 nulos preenchidos com "NAO_INFORMADO"; convertido para category',
     'Original (tratada)',
     'Segmentação por carreira; agrupamento em BI'),

    ('nivel',
     'Nível de escolaridade exigido (NS=Superior, NI=Intermediário, NM=Médio, NAO_INFORMADO)',
     'category',
     'Base SEGES — campo NIVEL',
     '1.255 nulos preenchidos com "NAO_INFORMADO"; convertido para category',
     'Original (tratada)',
     'Filtro; segmentação; análise comparativa entre níveis'),

    ('em_extincao',
     'Indica se o cargo está formalmente em extinção (S/N)',
     'category',
     'Base SEGES — campo CARGO_EM_EXTINCAO',
     'Renomeado; convertido para category',
     'Original (transformada)',
     'Filtro; segmentação; análise de carreiras descontinuadas'),

    # ───────── Bloco D. Quantitativos brutos ─────────
    ('qtd_aprovada',
     'Vagas aprovadas em lei para o cargo (teto legal)',
     'int64',
     'Base SEGES — campo APROVADA',
     'Renomeado de APROVADA',
     'Original',
     'Medida; teto legal; base para taxa_ocupacao'),

    ('qtd_distribuida',
     'Vagas distribuídas ao órgão pelo Órgão Central de Gestão de Pessoas',
     'int64',
     'Base SEGES — campo DISTRIBUIDA',
     'Renomeado de DISTRIBUIDA',
     'Original',
     'Medida; comparação com APROVADA revela margem operacional'),

    ('qtd_ocupada',
     'Vagas atualmente ocupadas por servidores ativos',
     'int64',
     'Base SEGES — campo OCUPADA',
     'Renomeado de OCUPADA',
     'Original',
     'Medida principal; numerador de taxa_ocupacao'),

    ('qtd_vaga',
     'Vagas em aberto, aguardando preenchimento',
     'int64',
     'Base SEGES — campo VAGA',
     'Renomeado de VAGA',
     'Original',
     'Medida principal; KPI de déficit operacional'),

    # ───────── Bloco E. Vacâncias por tipo ─────────
    ('vac_aposentadoria',
     'Vacâncias por aposentadoria no mês',
     'int64',
     'Base SEGES — VACANCIA_POR_APOSENTADORIA',
     'Renomeado para vac_aposentadoria',
     'Original',
     'Principal medida de fluxo (76% das vacâncias); planejamento previdenciário'),

    ('vac_posse_cargo_inac',
     'Vacâncias por posse em outro cargo público inacumulável (Lei 8.112/90)',
     'int64',
     'Base SEGES — VACANCIA_POR_POSSE_CARGO_INAC',
     'Renomeado para vac_posse_cargo_inac',
     'Original',
     'Mede mobilidade interna no funcionalismo'),

    ('vac_exoneracao',
     'Vacâncias por exoneração (voluntária ou de cargo em comissão)',
     'int64',
     'Base SEGES — VACANCIA_POR_EXONERACAO',
     'Renomeado para vac_exoneracao',
     'Original',
     'Mede rotatividade voluntária'),

    ('vac_falecimento',
     'Vacâncias por falecimento do servidor',
     'int64',
     'Base SEGES — VACANCIA_POR_FALECIMENTO',
     'Renomeado para vac_falecimento',
     'Original',
     'Proxy demográfico; correlaciona com idade do quadro'),

    ('vac_demissao',
     'Vacâncias por demissão (sanção administrativa)',
     'int64',
     'Base SEGES — VACANCIA_POR_DEMISSAO',
     'Renomeado para vac_demissao',
     'Original',
     'Indicador de governança e probidade'),

    ('vac_promocao',
     'Vacâncias por promoção a outro cargo do mesmo órgão',
     'int64',
     'Base SEGES — VACANCIA_POR_PROMOCAO',
     'Renomeado para vac_promocao',
     'Original',
     'Mede mobilidade vertical (raro: <0,2% do total)'),

    ('vac_readaptacao',
     'Vacâncias por readaptação funcional (laudo médico)',
     'int64',
     'Base SEGES — VACANCIA_POR_READAPTACAO',
     'Renomeado para vac_readaptacao',
     'Original',
     'Indicador de saúde ocupacional (categoria muito rara)'),

    # ───────── Bloco F. Features derivadas (quantitativas) ─────────
    ('total_vacancias',
     'Soma de todas as 7 vacâncias do mês',
     'int64',
     'Derivado das 7 colunas vac_*',
     'df[vac_*].sum(axis=1)',
     'Criada',
     'KPI consolidado de fluxo de saída; denominador de pct_aposentadoria'),

    ('deficit_nominal',
     'Quantidade absoluta de servidores faltantes (aprovada - ocupada)',
     'int64',
     'Derivado de qtd_aprovada e qtd_ocupada',
     'qtd_aprovada - qtd_ocupada',
     'Criada',
     'KPI de gestão; quantifica déficit em valor absoluto'),

    ('taxa_ocupacao',
     'Razão ocupada/aprovada (proxy de saúde do quadro)',
     'float64',
     'Derivado de qtd_ocupada e qtd_aprovada',
     'qtd_ocupada / qtd_aprovada; NaN se aprovada=0',
     'Criada',
     'KPI principal; análise comparativa entre órgãos e níveis'),

    ('taxa_vacancia_mensal',
     'Proporção de saída em relação ao quadro ativo no mês',
     'float64',
     'Derivado de total_vacancias e qtd_ocupada',
     'total_vacancias / qtd_ocupada; NaN se ocupada=0',
     'Criada',
     'KPI de rotatividade; identifica cargos em colapso'),

    ('pct_aposentadoria',
     'Proporção das vacâncias do mês que foram por aposentadoria',
     'float64',
     'Derivado de vac_aposentadoria e total_vacancias',
     'vac_aposentadoria / total_vacancias; NaN se total=0',
     'Criada',
     'Proxy de envelhecimento institucional; clustering de órgãos'),

    ('diferenca_distribuicao',
     'Diferença (ocupada + vaga) - distribuida; captura inconsistências',
     'int64',
     'Derivado de qtd_ocupada, qtd_vaga, qtd_distribuida',
     '(qtd_ocupada + qtd_vaga) - qtd_distribuida',
     'Criada',
     'Materializa o problema P08 do diagnóstico; cessões e comissionamentos'),

    # ───────── Bloco G. Features derivadas (categóricas) ─────────
    ('porte_cargo',
     'Faixa de tamanho do cargo: Micro(1-2), Pequeno(3-10), Médio(11-50), Grande(51-500), Mega(>500)',
     'category',
     'Derivado de qtd_aprovada',
     'pd.cut com cortes de negócio',
     'Criada',
     'Mining de regras; segmentação executiva; comunicação ao gestor'),

    ('faixa_taxa_ocupacao',
     'Status operacional: Crítica(<40%), Baixa(40-70%), Adequada(70-90%), Completa(90-100%), Excedente(>100%)',
     'category',
     'Derivado de taxa_ocupacao',
     'pd.cut com cortes operacionais',
     'Criada',
     'Mining individual; status visual para gestores'),

    # ───────── Bloco H. Versões normalizadas ─────────
    ('qtd_aprovada_robust',
     'qtd_aprovada normalizada com RobustScaler (mediana=0, IQR=1)',
     'float64',
     'Derivado de qtd_aprovada',
     'sklearn RobustScaler.fit_transform',
     'Criada',
     'Insumo para algoritmos de DM sensíveis a escala (K-Means, KNN, PCA)'),

    ('qtd_ocupada_robust',
     'qtd_ocupada normalizada com RobustScaler',
     'float64',
     'Derivado de qtd_ocupada',
     'sklearn RobustScaler.fit_transform',
     'Criada',
     'Insumo para algoritmos de DM'),

    ('qtd_vaga_robust',
     'qtd_vaga normalizada com RobustScaler',
     'float64',
     'Derivado de qtd_vaga',
     'sklearn RobustScaler.fit_transform',
     'Criada',
     'Insumo para algoritmos de DM'),

    ('total_vacancias_robust',
     'total_vacancias normalizada com RobustScaler',
     'float64',
     'Derivado de total_vacancias',
     'sklearn RobustScaler.fit_transform',
     'Criada',
     'Insumo para algoritmos de DM'),

    ('deficit_nominal_robust',
     'deficit_nominal normalizado com RobustScaler',
     'float64',
     'Derivado de deficit_nominal',
     'sklearn RobustScaler.fit_transform',
     'Criada',
     'Insumo para algoritmos de DM'),

    # ───────── Bloco I. Flags ─────────
    ('is_outlier',
     'Flag binária: 1 se qtd_aprovada é outlier pelo método IQR (limite: Q3+1.5*IQR)',
     'int64 (0/1)',
     'Derivado de qtd_aprovada',
     '(qtd_aprovada > Q3 + 1.5*IQR).astype(int)',
     'Criada',
     'Permite análises segmentadas entre cargos típicos e institucionais'),
]

print(f"Catálogo definido com {len(CATALOGO)} entradas.")

Catálogo definido com 37 entradas.


### 4.2 Validação: catálogo coincide exatamente com as colunas do dataset

In [10]:
# Extrair os nomes do catálogo (primeiro elemento de cada tupla)
nomes_catalogo = [c[0] for c in CATALOGO]
nomes_dataset  = list(df_final.columns)

# Conjuntos para comparação
set_cat = set(nomes_catalogo)
set_df  = set(nomes_dataset)

faltam_no_cat   = set_df - set_cat
sobram_no_cat   = set_cat - set_df
duplicadas_cat  = len(nomes_catalogo) - len(set_cat)

print(f"Colunas no dataset:  {len(nomes_dataset)}")
print(f"Entradas no catálogo: {len(nomes_catalogo)}")
print(f"Faltam no catálogo:  {faltam_no_cat or 'nenhuma'}")
print(f"Sobram no catálogo:  {sobram_no_cat or 'nenhuma'}")
print(f"Duplicadas no cat.:  {duplicadas_cat}")

assert not faltam_no_cat,  f'Catálogo incompleto: faltam {faltam_no_cat}'
assert not sobram_no_cat,  f'Catálogo tem extras: {sobram_no_cat}'
assert duplicadas_cat == 0, 'Catálogo tem entradas duplicadas'

print('\n✓ Catálogo e dataset estão perfeitamente alinhados.')

Colunas no dataset:  37
Entradas no catálogo: 37
Faltam no catálogo:  nenhuma
Sobram no catálogo:  nenhuma
Duplicadas no cat.:  0

✓ Catálogo e dataset estão perfeitamente alinhados.


### 4.3 Coleta dinâmica dos exemplos de valor

Em vez de hardcodar exemplos, extraímos um valor real de cada coluna do dataset. Isso garante que o catálogo nunca diverge dos dados.

In [11]:
def primeiro_valor_legivel(serie):
    """Retorna o primeiro valor não nulo da série em formato legível."""
    s = serie.dropna()
    if s.empty:
        return '(sem valor)'
    v = s.iloc[0]
    if isinstance(v, pd.Timestamp):
        return v.strftime('%Y-%m-%d')
    if isinstance(v, float):
        return f'{v:.3f}'
    return str(v)

exemplos = {c: primeiro_valor_legivel(df_final[c]) for c in df_final.columns}

print('Amostra dos exemplos coletados:')
for c in ['ano_mes', 'cod_orgao', 'taxa_ocupacao', 'porte_cargo', 'is_outlier']:
    print(f"  {c:<25} → {exemplos[c]}")

Amostra dos exemplos coletados:
  ano_mes                   → 2026-03-01
  cod_orgao                 → 13000
  taxa_ocupacao             → 1.000
  porte_cargo               → Micro (1-2)
  is_outlier                → 0


### 4.4 Construção do DataFrame do catálogo

In [12]:
# Montar o DataFrame do catálogo com 8 atributos (a coluna 'exemplo' vem do passo anterior)
linhas_catalogo = []
for (nome, descricao, tipo, origem, tratamento, origem_tipo, uso) in CATALOGO:
    linhas_catalogo.append({
        'nome_coluna':         nome,
        'descricao':           descricao,
        'tipo_dado':           tipo,
        'exemplo':             exemplos[nome],
        'origem':              origem,
        'tratamento_aplicado': tratamento,
        'origem_tipo':         origem_tipo,   # 'Original' / 'Original (transformada)' / 'Criada'
        'uso_esperado':        uso,
    })

cat_df = pd.DataFrame(linhas_catalogo)

print(f"DataFrame do catálogo: {cat_df.shape[0]} linhas × {cat_df.shape[1]} atributos\n")
print('Distribuição da origem das colunas:')
print(cat_df['origem_tipo'].value_counts().to_string())

DataFrame do catálogo: 37 linhas × 8 atributos

Distribuição da origem das colunas:
origem_tipo
Criada                     17
Original                   14
Original (transformada)     4
Original (tratada)          2


### 4.5 Exportação do catálogo como XLSX formatado

Geramos um arquivo Excel com **duas planilhas**:

1. **`Catalogo`** — o catálogo completo, com cabeçalho destacado e larguras ajustadas.
2. **`Metadados`** — informações sobre o projeto (versão, data de geração, fonte, totais).

Usamos `openpyxl` diretamente (em vez de só `pandas.to_excel`) para conseguir formatação profissional: cabeçalho em negrito, cores, freeze panes, larguras adequadas — atendendo às boas práticas exigidas pela disciplina.

In [13]:
# Criar o workbook
wb = Workbook()

# ───────── PLANILHA 1: Catálogo ─────────
ws_cat = wb.active
ws_cat.title = 'Catalogo'

# Estilos
COR_CABECALHO     = 'FF1F4E78'   # azul escuro
COR_TEXTO_CABEC   = 'FFFFFFFF'   # branco
COR_ZEBRA_PAR     = 'FFF2F2F2'   # cinza claro
FONTE_PADRAO      = 'Arial'

header_font  = Font(name=FONTE_PADRAO, size=11, bold=True, color=COR_TEXTO_CABEC)
header_fill  = PatternFill('solid', start_color=COR_CABECALHO, end_color=COR_CABECALHO)
header_align = Alignment(horizontal='center', vertical='center', wrap_text=True)

cell_font_pad  = Font(name=FONTE_PADRAO, size=10)
cell_align_pad = Alignment(horizontal='left', vertical='top', wrap_text=True)
borda_fina = Border(
    left=Side(style='thin', color='CCCCCC'),
    right=Side(style='thin', color='CCCCCC'),
    top=Side(style='thin', color='CCCCCC'),
    bottom=Side(style='thin', color='CCCCCC'),
)

# Cabeçalhos
cabecalhos = list(cat_df.columns)
for col_idx, header in enumerate(cabecalhos, start=1):
    cell = ws_cat.cell(row=1, column=col_idx, value=header)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_align
    cell.border = borda_fina

# Linhas de dados (com zebra)
for row_idx, registro in enumerate(cat_df.itertuples(index=False), start=2):
    for col_idx, valor in enumerate(registro, start=1):
        cell = ws_cat.cell(row=row_idx, column=col_idx, value=valor)
        cell.font = cell_font_pad
        cell.alignment = cell_align_pad
        cell.border = borda_fina
        if row_idx % 2 == 0:  # zebra
            cell.fill = PatternFill('solid', start_color=COR_ZEBRA_PAR,
                                    end_color=COR_ZEBRA_PAR)

# Larguras de colunas calibradas para legibilidade
larguras = {
    'nome_coluna':         22,
    'descricao':           60,
    'tipo_dado':           15,
    'exemplo':             20,
    'origem':              45,
    'tratamento_aplicado': 50,
    'origem_tipo':         22,
    'uso_esperado':        55,
}
for col_idx, header in enumerate(cabecalhos, start=1):
    ws_cat.column_dimensions[get_column_letter(col_idx)].width = larguras.get(header, 20)

# Altura da linha do cabeçalho
ws_cat.row_dimensions[1].height = 30

# Freeze na primeira linha
ws_cat.freeze_panes = 'A2'

print(f'Planilha "Catalogo" formatada com {ws_cat.max_row - 1} entradas de coluna.')

Planilha "Catalogo" formatada com 37 entradas de coluna.


In [14]:
# ───────── PLANILHA 2: Metadados ─────────
ws_meta = wb.create_sheet('Metadados')

metadados = [
    ('Projeto',                  'PM3 — Tratamento de Dados: Cargos Vagos e Vacâncias'),
    ('Base de origem',           'Cargos Vagos e Vacâncias do Poder Executivo Federal Civil'),
    ('Competência',              '03/2026 (snapshot mensal)'),
    ('Fonte primária',           'Portal Brasileiro de Dados Abertos — dados.gov.br'),
    ('Órgão responsável',        'SEGES / Ministério da Gestão e da Inovação em Serviços Públicos'),
    ('Arquivo original',         'CargosVagosVacancias_202603.ods (aba por_Órgao_e_Cargo)'),
    ('Linhas (registros)',       f'{df_final.shape[0]:,}'.replace(',', '.')),
    ('Colunas',                  str(df_final.shape[1])),
    ('Colunas originais',        str(sum(1 for c in CATALOGO if 'Original' in c[5]))),
    ('Colunas criadas',          str(sum(1 for c in CATALOGO if c[5] == 'Criada'))),
    ('Dicionário oficial',       'Dicionario_CargosVagosVacancias.pdf (repositorio.dados.gov.br)'),
    ('Licença',                  'CC BY 4.0 (Dados Abertos)'),
    ('Versão do catálogo',       '1.0'),
    ('Gerado por',               'Notebook 04 (04_dataset_final.ipynb)'),
    ('Data de geração',          pd.Timestamp.today().strftime('%Y-%m-%d %H:%M')),
]

# Cabeçalhos da aba metadados
ws_meta.cell(row=1, column=1, value='Atributo').font = header_font
ws_meta.cell(row=1, column=1).fill = header_fill
ws_meta.cell(row=1, column=1).alignment = header_align
ws_meta.cell(row=1, column=2, value='Valor').font = header_font
ws_meta.cell(row=1, column=2).fill = header_fill
ws_meta.cell(row=1, column=2).alignment = header_align

for i, (atributo, valor) in enumerate(metadados, start=2):
    c1 = ws_meta.cell(row=i, column=1, value=atributo)
    c2 = ws_meta.cell(row=i, column=2, value=str(valor))
    c1.font = Font(name=FONTE_PADRAO, size=10, bold=True)
    c2.font = Font(name=FONTE_PADRAO, size=10)
    c1.alignment = Alignment(horizontal='left', vertical='top')
    c2.alignment = Alignment(horizontal='left', vertical='top', wrap_text=True)
    if i % 2 == 0:
        c1.fill = PatternFill('solid', start_color=COR_ZEBRA_PAR, end_color=COR_ZEBRA_PAR)
        c2.fill = PatternFill('solid', start_color=COR_ZEBRA_PAR, end_color=COR_ZEBRA_PAR)

ws_meta.column_dimensions['A'].width = 26
ws_meta.column_dimensions['B'].width = 70
ws_meta.row_dimensions[1].height = 24
ws_meta.freeze_panes = 'A2'

print(f'Planilha "Metadados" formatada com {len(metadados)} atributos.')

Planilha "Metadados" formatada com 15 atributos.


In [15]:
# ───────── Salvar o workbook ─────────
DOCUMENTACAO.mkdir(parents=True, exist_ok=True)
caminho_catalogo = DOCUMENTACAO / 'catalogo_dados.xlsx'
wb.save(caminho_catalogo)

print(f"Catálogo salvo: {caminho_catalogo}")
print(f"  Tamanho: {caminho_catalogo.stat().st_size / 1024:.1f} KB")
print(f"  Planilhas: {wb.sheetnames}")

# Validação: relê e confere shape do catálogo
cat_check = pd.read_excel(caminho_catalogo, sheet_name='Catalogo')
meta_check = pd.read_excel(caminho_catalogo, sheet_name='Metadados')
print(f"\n  Releitura:")
print(f"  - Catalogo:  {cat_check.shape[0]} linhas × {cat_check.shape[1]} colunas")
print(f"  - Metadados: {meta_check.shape[0]} linhas × {meta_check.shape[1]} colunas")
assert cat_check.shape == (37, 8), 'Catálogo com shape inesperado após releitura'
print('  ✓ Catálogo validado.')

Catálogo salvo: /home/claude/PM3_CargosVagosVacancias_v2/documentacao/catalogo_dados.xlsx
  Tamanho: 10.5 KB
  Planilhas: ['Catalogo', 'Metadados']

  Releitura:
  - Catalogo:  37 linhas × 8 colunas
  - Metadados: 15 linhas × 2 colunas
  ✓ Catálogo validado.


## 5. Resumo final do projeto (panorama completo)

Esta seção consolida todos os entregáveis gerados ao longo dos cinco notebooks, organizados por etapa do enunciado.

### 5.1 Artefatos por etapa

| Etapa | Conteúdo | Notebook | Artefato gerado |
|---|---|---|---|
| **6.1** | Planejamento (tema, fonte, perguntas) | 00 | Markdown completo no notebook + 11 perguntas analíticas |
| **6.2** | Relação com BI, Big Data, Data Mining | 00 | Tabela-resumo de aplicações em 2.4 |
| **6.3** | Modelagem inicial dos dados | 01 | Inventário de colunas, tipos, dimensões/medidas |
| **6.4** | Diagnóstico da qualidade | 01 | `documentacao/problemas_qualidade.xlsx` (10 problemas) |
| **6.5** | Análise Exploratória | 02b | 7 gráficos PNG em `evidencias_aed/` |
| **6.6** | Seleção dos dados | 02 | Justificativa de manter 20 cols / 12.769 linhas |
| **6.7** | Limpeza e pré-processamento | 02 | Renomeação, conversão de tipos, padronização |
| **6.8** | Tratamento de valores faltantes | 02 | 1.618 nulos preenchidos com "NAO_INFORMADO" |
| **6.9** | Tratamento de outliers | 02 | Flag `is_outlier` para 1.879 cargos institucionais |
| **6.10** | Transformação dos dados | 02 | datetime, snake_case, category |
| **6.11** | Agregação | 03 | 3 tabelas em `dados_tratados/agregacoes/` |
| **6.12** | Normalização | 03 | 5 colunas `*_robust` (RobustScaler) |
| **6.13** | Discretização | 03 | `porte_cargo` + `faixa_taxa_ocupacao` |
| **6.14** | Feature Engineering | 03 | 9 novas variáveis derivadas |
| **6.15** | Dataset final | **04** | `dados_tratados/dataset_final_tratado.csv` (e `.xlsx`) |
| **6.16** | Catálogo de dados | **04** | `documentacao/catalogo_dados.xlsx` |
| **6.17** | DataOps / organização | (estrutura) | Pastas, README, notebooks reproduzíveis |

### 5.2 Crescimento da base ao longo do tratamento

| Estágio | Linhas | Colunas | Memória | Nulos |
|---|---|---|---|---|
| Base bruta (`.ods`) | 12.769 | 20 | ~3 MB | 1.618 |
| Pós-limpeza (Notebook 02) | 12.769 | 21 | ~3 MB | 0 |
| Com features (Notebook 03) | 12.769 | 37 | ~6 MB | ~3 colunas com NaN justificado |
| Dataset final (Notebook 04) | 12.769 | 37 | ~6 MB | mesmos NaN justificados |

A base **não perdeu nenhum registro** ao longo do tratamento — todas as decisões privilegiaram **preservação de informação** sobre simplificação.


In [16]:
# Demonstração final: localização de todos os entregáveis
print("=== ENTREGÁVEIS DO PROJETO ===\n")

estrutura = {
    'dados_brutos/': [
        'CargosVagosVacancias_202603.ods   (original preservado)',
        'CargosVagosVacancias_202603.csv   (versão CSV)',
    ],
    'dados_tratados/': [
        'dataset_pos_limpeza.csv           (checkpoint Notebook 02)',
        'dataset_com_features.csv          (checkpoint Notebook 03)',
        'dataset_final_tratado.csv         (entregável Notebook 04)',
        'dataset_final_tratado.xlsx        (entregável Notebook 04)',
    ],
    'dados_tratados/agregacoes/': [
        'agg_por_orgao.csv',
        'agg_por_nivel.csv',
        'agg_por_tipo_vacancia.csv',
    ],
    'documentacao/': [
        'problemas_qualidade.xlsx          (Notebook 01)',
        'catalogo_dados.xlsx               (Notebook 04)',
    ],
    'evidencias_aed/': [
        '7 gráficos PNG (Notebook 02b)',
    ],
    'notebooks/': [
        '00_planejamento_e_contexto.ipynb',
        '01_diagnostico_qualidade.ipynb',
        '02_limpeza_e_transformacao.ipynb',
        '02b_analise_exploratoria.ipynb',
        '03_feature_engineering.ipynb',
        '04_dataset_final.ipynb',
    ],
}

for pasta, arquivos in estrutura.items():
    print(f"📁 {pasta}")
    for a in arquivos:
        print(f"   - {a}")
    print()

=== ENTREGÁVEIS DO PROJETO ===

📁 dados_brutos/
   - CargosVagosVacancias_202603.ods   (original preservado)
   - CargosVagosVacancias_202603.csv   (versão CSV)

📁 dados_tratados/
   - dataset_pos_limpeza.csv           (checkpoint Notebook 02)
   - dataset_com_features.csv          (checkpoint Notebook 03)
   - dataset_final_tratado.csv         (entregável Notebook 04)
   - dataset_final_tratado.xlsx        (entregável Notebook 04)

📁 dados_tratados/agregacoes/
   - agg_por_orgao.csv
   - agg_por_nivel.csv
   - agg_por_tipo_vacancia.csv

📁 documentacao/
   - problemas_qualidade.xlsx          (Notebook 01)
   - catalogo_dados.xlsx               (Notebook 04)

📁 evidencias_aed/
   - 7 gráficos PNG (Notebook 02b)

📁 notebooks/
   - 00_planejamento_e_contexto.ipynb
   - 01_diagnostico_qualidade.ipynb
   - 02_limpeza_e_transformacao.ipynb
   - 02b_analise_exploratoria.ipynb
   - 03_feature_engineering.ipynb
   - 04_dataset_final.ipynb



## 6. Conclusão

### O que foi feito neste notebook

1. **Etapa 6.15 (Dataset final):** validamos 6 critérios de qualidade automatizados, reordenamos as 37 colunas em 9 blocos semânticos, e exportamos o dataset final em CSV e XLSX. Os nulos remanescentes (em 3 features derivadas) foram justificados e preservados.

2. **Etapa 6.16 (Catálogo):** documentamos os 8 atributos exigidos pelo enunciado para cada uma das 37 colunas. O catálogo foi exportado como `catalogo_dados.xlsx` com:
   - Planilha **Catálogo** formatada profissionalmente (cabeçalho destacado, larguras calibradas, zebra rows, freeze panes).
   - Planilha **Metadados** com informações de proveniência, versão e estatísticas do dataset.

### Características do dataset final

- **12.769 registros** preservados desde a base bruta (zero perda).
- **37 colunas** organizadas em 9 blocos lógicos.
- **0 nulos não justificados** — apenas em 3 features derivadas onde o NaN é semanticamente correto.
- **0 duplicatas** pela chave funcional (cod_orgao + cod_cargo).
- **Tipos de dados corretos** em todas as colunas (datetime, string, category, int64, float64).
- **Reprodutível** — qualquer execução do pipeline (notebooks 02 → 03 → 04) regenera exatamente este dataset.

### O que falta para encerrar o projeto

| Pendência | Onde |
|---|---|
| Relatório final em Word | `documentacao/relatorio_final.docx` — montar com base nos markdowns dos notebooks |
| (Opcional) Fluxo Orange | `orange/fluxo_aed.ows` — gerar no Orange Data Mining seguindo o `GUIA_ORANGE.md` |

### Como citar este dataset

> SEGES/MGI. **Cargos Vagos e Vacâncias do Poder Executivo Federal Civil — competência 03/2026.** Dados originais disponíveis em `dados.gov.br`. Versão tratada produzida no PM3 (Tratamento de Dados), disciplina de Big Data Analytics / BI / Data Mining, 2026. Pipeline de tratamento documentado em 6 notebooks Jupyter; catálogo de dados em `catalogo_dados.xlsx`.
